In [ ]:
# Example 1: Basic Multiple Dispatch with Type-Based Function Overloading
from multipledispatch import dispatch
import numpy as np

# Create a dispatched function for different data processing operations
@dispatch(int, int)
def process_data(x, y):
    """Process two integers"""
    result = x + y
    print(f"Processing integers: {x} + {y} = {result}")
    return result

@dispatch(float, float)
def process_data(x, y):
    """Process two floats with precision"""
    result = x * y
    print(f"Processing floats: {x} * {y} = {result:.3f}")
    return result

@dispatch(str, str)
def process_data(x, y):
    """Process two strings"""
    result = f"{x}_{y}".upper()
    print(f"Processing strings: '{x}' + '{y}' -> '{result}'")
    return result

@dispatch(list, list)
def process_data(x, y):
    """Process two lists"""
    result = x + y  # Concatenate lists
    print(f"Processing lists: {len(x)} items + {len(y)} items = {len(result)} items")
    return result

@dispatch(int, str)
def process_data(x, y):
    """Process mixed int and string"""
    result = y * x
    print(f"Processing int + string: {x} * '{y}' = '{result}'")
    return result

print("Basic multiple dispatch demonstration:")
print("-" * 40)

# Test different type combinations
test_cases = [
    (5, 10),                    # integers
    (3.14, 2.5),               # floats
    ("hello", "world"),         # strings
    ([1, 2], [3, 4, 5]),       # lists
    (3, "abc"),                 # mixed types
]

for x, y in test_cases:
    try:
        result = process_data(x, y)
        print(f"Result type: {type(result).__name__}")
    except NotImplementedError as e:
        print(f"No dispatch found for {type(x).__name__}, {type(y).__name__}")
    print()

# Show dispatch information
print("Available dispatches for process_data:")
print(process_data.funcs)
print("Basic multiple dispatch completed successfully!")

In [ ]:
# Example 2: Advanced Type Hierarchies and Complex Dispatch Patterns
from multipledispatch import dispatch
import numbers
from collections.abc import Sequence, Mapping

# Define custom classes for demonstration
class Vector:
    def __init__(self, data):
        self.data = list(data)
    
    def __repr__(self):
        return f"Vector({self.data})"

class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(data)
        self.cols = len(data[0]) if data else 0
    
    def __repr__(self):
        return f"Matrix({self.rows}x{self.cols})"

# Complex dispatch patterns using type hierarchies
@dispatch(numbers.Number, numbers.Number)
def compute(a, b):
    """Generic numeric computation"""
    return f"Numbers: {a} + {b} = {a + b}"

@dispatch(numbers.Integral, numbers.Integral) 
def compute(a, b):
    """Specialized for integers (more specific than Number)"""
    return f"Integers: {a} ** {b} = {a ** b}"

@dispatch(numbers.Real, numbers.Real)
def compute(a, b):
    """Specialized for real numbers"""
    return f"Reals: {a} / {b} = {a / b:.3f}"

@dispatch(Sequence, Sequence)
def compute(a, b):
    """Handle any sequence type (list, tuple, string)"""
    return f"Sequences: len({len(a)}) + len({len(b)}) = {len(a) + len(b)}"

@dispatch(Mapping, Mapping)
def compute(a, b):
    """Handle any mapping type (dict, etc.)"""
    combined_keys = set(a.keys()) | set(b.keys())
    return f"Mappings: combined {len(combined_keys)} unique keys"

@dispatch(Vector, Vector)
def compute(a, b):
    """Vector addition"""
    if len(a.data) != len(b.data):
        return "Vector dimension mismatch"
    result = [x + y for x, y in zip(a.data, b.data)]
    return f"Vector addition: {Vector(result)}"

@dispatch(Matrix, Matrix)
def compute(a, b):
    """Matrix operations"""
    if a.rows != b.rows or a.cols != b.cols:
        return "Matrix dimension mismatch"
    return f"Matrix addition: {a.rows}x{a.cols} + {b.rows}x{b.cols}"

@dispatch(Vector, numbers.Number)
def compute(v, scalar):
    """Scalar multiplication of vector"""
    result = [x * scalar for x in v.data]
    return f"Vector scaling: {Vector(result)}"

@dispatch(Matrix, numbers.Number)
def compute(m, scalar):
    """Scalar multiplication of matrix"""
    return f"Matrix scaling: {m} * {scalar}"

print("Advanced type hierarchies and dispatch patterns:")
print("=" * 50)

# Test cases covering various type hierarchies
test_cases = [
    # Basic numbers (will use most specific match)
    (3, 4),                     # Integral -> uses integer dispatch
    (3.14, 2.5),               # Real -> uses real dispatch
    (2+3j, 1+1j),              # Complex -> uses generic Number dispatch
    
    # Sequences
    ([1, 2, 3], [4, 5]),       # Lists
    ((1, 2), (3, 4, 5)),       # Tuples
    ("hello", "world"),         # Strings (also sequences)
    
    # Mappings
    ({'a': 1, 'b': 2}, {'c': 3, 'b': 4}),  # Dictionaries
    
    # Custom objects
    (Vector([1, 2, 3]), Vector([4, 5, 6])),     # Vector addition
    (Matrix([[1, 2], [3, 4]]), Matrix([[5, 6], [7, 8]])),  # Matrix
    
    # Mixed custom and built-in
    (Vector([1, 2, 3]), 2),     # Vector * scalar
    (Matrix([[1, 2], [3, 4]]), 3),  # Matrix * scalar
]

for i, (a, b) in enumerate(test_cases, 1):
    print(f"{i:2d}. {type(a).__name__} + {type(b).__name__}:")
    try:
        result = compute(a, b)
        print(f"    {result}")
    except Exception as e:
        print(f"    Error: {e}")
        raise
    print()

# Demonstrate dispatch resolution
print("Dispatch resolution order example:")
print(f"Integer dispatch: {compute(5, 3)}")           # Most specific
print(f"Float dispatch: {compute(5.0, 3.0)}")         # Real numbers
print(f"Complex dispatch: {compute(5+0j, 3+0j)}")     # Falls back to Number

# Show all registered dispatches
print(f"\nTotal dispatches registered: {len(compute.funcs)}")
print("Advanced dispatch patterns completed!")

In [ ]:
# Example 3: Namespace Management and Dispatch Groups
from multipledispatch import Dispatcher

# Create separate dispatchers for different domains
math_ops = Dispatcher('math_ops')
string_ops = Dispatcher('string_ops')
data_ops = Dispatcher('data_ops')

# Mathematical operations dispatcher
@math_ops.register(int, int)
def add(x, y):
    """Integer addition with overflow protection"""
    result = x + y
    return f"Math: {x} + {y} = {result} (int)"

@math_ops.register(float, float)  
def add(x, y):
    """Float addition with precision handling"""
    result = x + y
    return f"Math: {x} + {y} = {result:.6f} (float)"

@math_ops.register(complex, complex)
def add(x, y):
    """Complex number addition"""
    result = x + y
    return f"Math: {x} + {y} = {result} (complex)"

# String operations dispatcher
@string_ops.register(str, str)
def add(x, y):
    """String concatenation"""
    result = x + y
    return f"String: '{x}' + '{y}' = '{result}'"

@string_ops.register(str, int)
def add(s, n):
    """String repetition"""
    result = s * n
    return f"String: '{s}' * {n} = '{result}'"

@string_ops.register(str, str, str)
def add(x, sep, y):
    """String join with separator"""
    result = x + sep + y
    return f"String join: '{x}' + '{sep}' + '{y}' = '{result}'"

# Data operations dispatcher
@data_ops.register(list, list)
def add(x, y):
    """List concatenation with metadata"""
    result = x + y
    return f"Data: list({len(x)}) + list({len(y)}) = list({len(result)})"

@data_ops.register(dict, dict)
def add(x, y):
    """Dictionary merging"""
    result = {**x, **y}
    return f"Data: dict merge {len(x)} + {len(y)} keys = {len(result)} keys"

@data_ops.register(set, set)
def add(x, y):
    """Set union"""
    result = x | y
    return f"Data: set({len(x)}) ∪ set({len(y)}) = set({len(result)})"

print("Namespace management and dispatch groups:")
print("=" * 45)

# Test each dispatcher independently
print("1. Mathematical operations:")
math_test_cases = [
    (5, 3),
    (2.5, 1.5),
    (1+2j, 3+1j)
]

for x, y in math_test_cases:
    result = math_ops(x, y)
    print(f"   {result}")

print("\n2. String operations:")
string_test_cases = [
    ("hello", "world"),
    ("abc", 3),
    ("hello", "-", "world")  # Three argument version
]

for args in string_test_cases:
    result = string_ops(*args)
    print(f"   {result}")

print("\n3. Data structure operations:")
data_test_cases = [
    ([1, 2], [3, 4, 5]),
    ({'a': 1, 'b': 2}, {'c': 3, 'b': 99}),
    ({1, 2, 3}, {3, 4, 5})
]

for x, y in data_test_cases:
    result = data_ops(x, y)
    print(f"   {result}")

# Demonstrate namespace isolation
print("\n4. Namespace isolation demonstration:")
print("Available functions in each dispatcher:")
print(f"   math_ops: {len(math_ops.funcs)} functions")
print(f"   string_ops: {len(string_ops.funcs)} functions") 
print(f"   data_ops: {len(data_ops.funcs)} functions")

# Show what happens when calling with unsupported types
print("\n5. Error handling in namespaced dispatchers:")
try:
    math_ops("hello", "world")  # String to math dispatcher
except NotImplementedError as e:
    print("   ✓ NotImplementedError caught for invalid type in math_ops")

try:
    string_ops([1, 2], [3, 4])  # List to string dispatcher  
except NotImplementedError as e:
    print("   ✓ NotImplementedError caught for invalid type in string_ops")

# Create a unified dispatcher that delegates to specialized ones
unified_dispatcher = Dispatcher('unified')

@unified_dispatcher.register(object, object)
def unified_add(x, y):
    """Unified dispatcher that delegates to specialized dispatchers"""
    # Try each dispatcher in order of preference
    for dispatcher_name, dispatcher in [
        ('math', math_ops),
        ('string', string_ops), 
        ('data', data_ops)
    ]:
        try:
            result = dispatcher(x, y)
            return f"[{dispatcher_name}] {result}"
        except NotImplementedError:
            continue
    
    return f"No handler found for {type(x).__name__} + {type(y).__name__}"

print("\n6. Unified dispatcher delegation:")
unified_test_cases = [
    (10, 20),              # Math
    ("foo", "bar"),        # String
    ([1, 2], [3]),         # Data
    (None, None)           # No handler
]

for x, y in unified_test_cases:
    result = unified_dispatcher(x, y)
    print(f"   {result}")

print("\nNamespace management completed successfully!")

In [ ]:
# Example 4: Union Types and Abstract Base Class Dispatch
from multipledispatch import dispatch
from collections.abc import Sequence, Mapping, Iterable
from numbers import Number, Integral, Real

# Union type dispatch - handle multiple types with one function
@dispatch((int, float))
def process_number(x):
    """Process either int or float"""
    return f"Number: {x} (type: {type(x).__name__})"

@dispatch((list, tuple))
def process_sequence(seq):
    """Process either list or tuple"""
    return f"Sequence: {len(seq)} items, first={seq[0] if seq else None}"

@dispatch((dict, set))
def process_collection(coll):
    """Process either dict or set"""
    if isinstance(coll, dict):
        return f"Dict: {len(coll)} keys"
    else:
        return f"Set: {len(coll)} items"

# Multiple argument union types
@dispatch((int, float), (int, float))
def compute(x, y):
    """Compute with any numeric types"""
    return f"Numeric: {x} * {y} = {x * y}"

@dispatch(str, (int, float))
def compute(s, n):
    """String repeated by number"""
    return f"String: '{s}' * {n} = '{s * int(n)}'"

@dispatch((list, tuple), (list, tuple))
def compute(seq1, seq2):
    """Concatenate sequences"""
    result = list(seq1) + list(seq2)
    return f"Sequences: {len(seq1)} + {len(seq2)} = {len(result)} items"

print("Union types and abstract base class dispatch:")
print("=" * 48)

# Test union type dispatch
print("1. Single argument union types:")
union_test_cases = [
    42,              # int
    3.14,            # float
    [1, 2, 3],       # list
    (4, 5, 6),       # tuple
    {'a': 1, 'b': 2},# dict
    {7, 8, 9}        # set
]

for item in union_test_cases:
    if isinstance(item, (int, float)):
        result = process_number(item)
    elif isinstance(item, (list, tuple)):
        result = process_sequence(item)
    elif isinstance(item, (dict, set)):
        result = process_collection(item)
    print(f"   {result}")

print("\n2. Multiple argument union types:")
compute_test_cases = [
    (5, 3),              # int, int
    (2.5, 4),            # float, int
    (3, 1.5),            # int, float
    ("abc", 3),          # str, int
    ("test", 2.7),       # str, float
    ([1, 2], [3, 4]),    # list, list
    ((1, 2), [3, 4]),    # tuple, list
]

for x, y in compute_test_cases:
    result = compute(x, y)
    print(f"   {type(x).__name__} × {type(y).__name__}: {result}")

# Abstract base class dispatch
print("\n3. Abstract base class dispatch:")

@dispatch(Number)
def analyze(x):
    """Handle any numeric type via ABC"""
    return f"Number ABC: {x} is numeric"

@dispatch(Sequence)
def analyze(x):
    """Handle any sequence type via ABC"""
    return f"Sequence ABC: length={len(x)}, indexable"

@dispatch(Mapping)
def analyze(x):
    """Handle any mapping type via ABC"""
    return f"Mapping ABC: {len(x)} mappings"

@dispatch(Iterable)
def analyze(x):
    """Handle any iterable (fallback)"""
    try:
        items = list(x)
        return f"Iterable ABC: {len(items)} items"
    except:
        return "Iterable ABC: not convertible to list"

# Test abstract base class dispatch
abc_test_cases = [
    42,                      # Number
    3.14,                    # Number
    [1, 2, 3],               # Sequence
    (4, 5, 6),               # Sequence
    "hello",                 # Sequence (str is sequence)
    {'a': 1, 'b': 2},        # Mapping
    {1, 2, 3},               # Iterable (set)
    range(5),                # Iterable (range)
]

for item in abc_test_cases:
    result = analyze(item)
    print(f"   {type(item).__name__:12s} -> {result}")

# Complex type hierarchies with union and ABC
print("\n4. Complex type hierarchies:")

@dispatch((Integral, Real), str)
def format_data(num, format_type):
    """Format numbers with string format type"""
    if format_type == "binary":
        return f"Binary: {bin(int(num))}"
    elif format_type == "hex":
        return f"Hex: {hex(int(num))}"
    elif format_type == "float":
        return f"Float: {float(num):.2f}"
    else:
        return f"Default: {num}"

@dispatch(Sequence, str)
def format_data(seq, format_type):
    """Format sequences with string format type"""
    if format_type == "csv":
        return f"CSV: {','.join(map(str, seq))}"
    elif format_type == "json":
        return f"JSON: [{', '.join(map(str, seq))}]"
    else:
        return f"Default: {seq}"

@dispatch(Mapping, str)
def format_data(mapping, format_type):
    """Format mappings with string format type"""
    if format_type == "keys":
        return f"Keys: {list(mapping.keys())}"
    elif format_type == "values":
        return f"Values: {list(mapping.values())}"
    else:
        return f"Default: {len(mapping)} items"

format_test_cases = [
    (42, "binary"),
    (255, "hex"),
    (3.14159, "float"),
    ([1, 2, 3, 4], "csv"),
    ([10, 20, 30], "json"),
    ({'a': 1, 'b': 2, 'c': 3}, "keys"),
    ({'x': 10, 'y': 20}, "values"),
]

for data, fmt in format_test_cases:
    result = format_data(data, fmt)
    print(f"   {type(data).__name__:12s} + '{fmt:6s}' -> {result}")

# Variadic dispatch with union types
print("\n5. Variadic signatures with union types:")

@dispatch((int, float))
def calculate(*args):
    """Sum multiple numbers"""
    total = sum(args)
    return f"Sum of {len(args)} numbers: {total}"

@dispatch(str)
def calculate(*args):
    """Concatenate multiple strings"""
    result = ''.join(args)
    return f"Joined {len(args)} strings: '{result}'"

# Note: multipledispatch doesn't support true variadic dispatch on types
# So we test single-type dispatch instead
single_arg_tests = [
    (42,),
    (3.14,),
    ("hello",),
]

for args in single_arg_tests:
    result = calculate(*args)
    print(f"   {type(args[0]).__name__}: {result}")

# Demonstrate dispatch resolution order
print("\n6. Dispatch resolution with type hierarchy:")

@dispatch(object)
def type_check(x):
    """Most general handler"""
    return "object (most general)"

@dispatch(Number)
def type_check(x):
    """Number handler (more specific)"""
    return "Number (more specific)"

@dispatch(Integral)
def type_check(x):
    """Integral handler (most specific)"""
    return "Integral (most specific)"

resolution_tests = [
    int(42),        # Matches Integral
    float(3.14),    # Matches Number
    complex(1+2j),  # Matches Number
    "string",       # Matches object
]

for item in resolution_tests:
    result = type_check(item)
    print(f"   {type(item).__name__:12s} -> {result}")

print("\n7. Namespace isolation with union types:")
from multipledispatch import Dispatcher

math_ns = Dispatcher('math')
text_ns = Dispatcher('text')

@math_ns.register((int, float), (int, float))
def operation(a, b):
    return f"Math: {a} + {b} = {a + b}"

@text_ns.register(str, str)
def operation(a, b):
    return f"Text: '{a}' + '{b}' = '{a + b}'"

@text_ns.register((str, ), (int, float))
def operation(a, b):
    return f"Text: '{a}' repeated {b} times = '{a * int(b)}'"

namespace_tests = [
    (math_ns, 10, 20),
    (math_ns, 3.5, 2.5),
    (text_ns, "hello", "world"),
    (text_ns, "test", 3),
]

for ns, a, b in namespace_tests:
    result = ns(a, b)
    print(f"   {ns.name}: {result}")

print("\nUnion types and ABC dispatch completed successfully!")

In [ ]:
# Example 5: Production Design Patterns and Performance Considerations
from multipledispatch import dispatch, Dispatcher
import time
import functools

# Create a production-ready API dispatcher system
api_dispatcher = Dispatcher('api_handler')

# Base API response structure
class APIResponse:
    def __init__(self, data, status_code=200, message="Success"):
        self.data = data
        self.status_code = status_code
        self.message = message
        self.timestamp = time.time()
    
    def to_dict(self):
        return {
            'data': self.data,
            'status_code': self.status_code,
            'message': self.message,
            'timestamp': self.timestamp
        }

# User data model
class User:
    def __init__(self, user_id, name, email):
        self.user_id = user_id
        self.name = name
        self.email = email
    
    def to_dict(self):
        return {'user_id': self.user_id, 'name': self.name, 'email': self.email}

class Product:
    def __init__(self, product_id, name, price):
        self.product_id = product_id
        self.name = name
        self.price = price
    
    def to_dict(self):
        return {'product_id': self.product_id, 'name': self.name, 'price': self.price}

# Performance monitoring decorator
def monitor_dispatch(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        
        # Add performance metadata to response
        if isinstance(result, APIResponse):
            # Handle case where data might be None
            if result.data is None:
                result.data = {}
            elif not isinstance(result.data, dict):
                # If data is not a dict, wrap it
                result.data = {'value': result.data}
            
            result.data['_performance'] = {
                'execution_time': end_time - start_time,
                'function': func.__name__
            }
        return result
    return wrapper

# API handlers using multiple dispatch
@api_dispatcher.register(User, str)
@monitor_dispatch
def handle_request(user, action):
    """Handle user-related requests"""
    if action == "profile":
        return APIResponse(user.to_dict(), message="User profile retrieved")
    elif action == "delete":
        return APIResponse({'deleted': True}, message="User deleted")
    else:
        return APIResponse({'error': f"Unknown user action: {action}"}, 400, f"Unknown user action: {action}")

@api_dispatcher.register(Product, str)
@monitor_dispatch  
def handle_request(product, action):
    """Handle product-related requests"""
    if action == "details":
        return APIResponse(product.to_dict(), message="Product details retrieved")
    elif action == "purchase":
        return APIResponse({'purchased': True, 'total': product.price}, 
                          message="Product purchased")
    else:
        return APIResponse({'error': f"Unknown product action: {action}"}, 400, f"Unknown product action: {action}")

@api_dispatcher.register(list, str)
@monitor_dispatch
def handle_request(items, action):
    """Handle bulk operations on lists"""
    if action == "count":
        return APIResponse({'count': len(items)}, message="Items counted")
    elif action == "validate":
        valid_items = [item for item in items if hasattr(item, 'to_dict')]
        return APIResponse({
            'total': len(items),
            'valid': len(valid_items),
            'invalid': len(items) - len(valid_items)
        }, message="Items validated")
    else:
        return APIResponse({'error': f"Unknown bulk action: {action}"}, 400, f"Unknown bulk action: {action}")

@api_dispatcher.register(dict, str)
@monitor_dispatch
def handle_request(config, action):
    """Handle configuration requests"""
    if action == "validate":
        required_keys = ['version', 'environment']
        missing = [key for key in required_keys if key not in config]
        is_valid = len(missing) == 0
        
        return APIResponse({
            'valid': is_valid,
            'missing_keys': missing,
            'config_size': len(config)
        }, message="Configuration validated")
    elif action == "backup":
        return APIResponse({'backed_up': True, 'keys': list(config.keys())},
                          message="Configuration backed up")
    else:
        return APIResponse({'error': f"Unknown config action: {action}"}, 400, f"Unknown config action: {action}")

# Error handling dispatcher
error_handler = Dispatcher('error_handler')

@error_handler.register(Exception, str)
def handle_error(error, context):
    """Generic error handler"""
    return APIResponse(
        {'error_type': type(error).__name__, 'context': context},
        status_code=500,
        message=f"Internal error: {str(error)[:100]}"
    )

@error_handler.register(ValueError, str)
def handle_error(error, context):
    """Value error handler"""
    return APIResponse(
        {'validation_error': str(error), 'context': context},
        status_code=400,
        message="Validation failed"
    )

@error_handler.register(KeyError, str)
def handle_error(error, context):
    """Key error handler"""
    return APIResponse(
        {'missing_key': str(error), 'context': context},
        status_code=404,
        message="Required key not found"
    )

print("Production design patterns and performance considerations:")
print("=" * 58)

# Test the production API system
print("1. API request handling with multiple dispatch:")
test_requests = [
    (User(1, "Alice", "alice@example.com"), "profile"),
    (User(2, "Bob", "bob@example.com"), "delete"),
    (Product(101, "Laptop", 999.99), "details"),
    (Product(102, "Mouse", 29.99), "purchase"),
    ([User(1, "Alice", "alice@example.com"), User(2, "Bob", "bob@example.com")], "count"),
    ({'version': '1.0', 'environment': 'production', 'debug': False}, "validate"),
    (User(3, "Charlie", "charlie@example.com"), "invalid_action")
]

for obj, action in test_requests:
    response = api_dispatcher(obj, action)
    result_dict = response.to_dict()
    
    print(f"   {type(obj).__name__} + '{action}':")
    print(f"     Status: {result_dict['status_code']}")
    print(f"     Message: {result_dict['message']}")
    if result_dict['data'] and '_performance' in result_dict['data']:
        perf = result_dict['data']['_performance']
        print(f"     Performance: {perf['execution_time']:.6f}s")
    print()

# Test error handling
print("2. Error handling with dispatch:")
error_test_cases = [
    (ValueError("Invalid input format"), "data_validation"),
    (KeyError("'required_field'"), "config_parsing"),
    (RuntimeError("Database connection failed"), "database_operation")
]

for error, context in error_test_cases:
    response = error_handler(error, context)
    result_dict = response.to_dict()
    
    print(f"   {type(error).__name__}: {str(error)}")
    print(f"     Status: {result_dict['status_code']}")
    print(f"     Message: {result_dict['message']}")
    print()

# Performance benchmark
print("3. Performance benchmarking:")

def benchmark_dispatch_vs_traditional(iterations=1000):
    """Compare dispatch performance vs traditional approach"""
    
    # Traditional approach using if/elif
    def traditional_handler(obj, action):
        if isinstance(obj, User):
            if action == "profile":
                return obj.to_dict()
            elif action == "delete":
                return {'deleted': True}
        elif isinstance(obj, Product):
            if action == "details":
                return obj.to_dict()
            elif action == "purchase":
                return {'purchased': True, 'total': obj.price}
        return None
    
    # Test data
    test_user = User(1, "Test", "test@example.com")
    test_product = Product(1, "Test Product", 50.0)
    
    # Benchmark dispatch approach
    start_time = time.time()
    for _ in range(iterations):
        api_dispatcher(test_user, "profile")
        api_dispatcher(test_product, "details")
    dispatch_time = time.time() - start_time
    
    # Benchmark traditional approach
    start_time = time.time() 
    for _ in range(iterations):
        traditional_handler(test_user, "profile")
        traditional_handler(test_product, "details")
    traditional_time = time.time() - start_time
    
    return dispatch_time, traditional_time

dispatch_time, traditional_time = benchmark_dispatch_vs_traditional(1000)

print(f"   Dispatch approach: {dispatch_time:.6f}s")
print(f"   Traditional approach: {traditional_time:.6f}s")
print(f"   Overhead ratio: {dispatch_time / traditional_time:.2f}x")

# Show registered functions
print(f"\n4. System introspection:")
print(f"   API handlers registered: {len(api_dispatcher.funcs)}")
print(f"   Error handlers registered: {len(error_handler.funcs)}")

print("   Dispatch signature coverage:")
for signature, func in list(api_dispatcher.funcs.items())[:5]:
    types = [t.__name__ if hasattr(t, '__name__') else str(t) for t in signature]
    print(f"     {' × '.join(types)} -> {func.__name__}")

print("\nProduction design patterns completed successfully!")
print("Multiple dispatch provides clean, maintainable polymorphic APIs!")